# T04 – Vocals, keys & swing in 2023
*MIR & ML for Audio · TU Berlin · WiSe 2026/27*

| | |
|---|---|
| **Name** | Hallvard Jensen |
| **Student ID** | 0549158 |
| **Corpus (genre)** | Nu Disco/Disco |

**Submission:** work in this notebook – code, figures and text answers all stay here. Before uploading: **Restart Kernel & Run All**, check that every output is shown, then upload this `.ipynb` file.

**Documentation:** document your work directly in the notebook – comments in your code cells and, where helpful, short markdown cells.

### Sources & AI tools
- **Sources** (literature, data, code):
- **LLM coding assistants** (tool/model – how and what for):

## Tasks
1. Investigate the use of vocals in your EDMC genre in 2023!
   - In what percentage of the 2023 Top 100 tracks do vocals occur?
   - What share do the different keys have among the vocals in tracks from 2023?
   - Make it possible to follow by ear why the key could not be detected for certain tracks
   - Create a results plot and describe the results in one sentence a layperson can understand!
2. Try to determine the typical use of swing in the drums in your EDMC genre in 2023!
   - *By “swing” we mean periodically recurring timing deviations of paired eighth-note onsets following a general pattern (50–66.7 %, triplet swing being the absolute maximum).*
   - *Tip: such patterns can be observed in the inter-onset tempogram.*
   - Create a histogram of the degree of swing determined for the 2023 Top 100 tracks of your corpus
   - Identify the 2023 track with a) the strongest swing and b) the weakest swing
   - Show the logic of your swing detection for these two tracks
   - Describe the results in one sentence a layperson can understand
   - Check the plausibility of your approach with listening examples

---
## Setup

Expensive steps (feature extraction, separation, model training): save the results to `OUTPUT_DIR` (e.g. as CSV) and load them on the next run instead of recomputing – *Run All* stays fast. Results of earlier sessions are in `output/T02/`, `output/T03/`, …

In [73]:
# import standard packages you'll need

import sys
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(sys.executable)

# ─ add your own additional imports here:
import demucs.api
import soundfile as sf
import librosa
import numpy.typing as npt
import tqdm
from IPython.display import Audio, display

c:\Users\Hallvard\miniforge3\envs\mirml\python.exe


In [47]:
# ── CONFIG ───────────────────────────────────────────────────────────────
CORPUS_DIR = Path("data") / "nu-disco-disco"   # ← name of your corpus folder
# ─────────────────────────────────────────────────────────────────────────
SESSION = "T04"
PREVIEW_DIR = CORPUS_DIR / "previews"
OUTPUT_DIR = Path("output") / SESSION      # results of this session
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert CORPUS_DIR.is_dir(), f"Not found: {CORPUS_DIR.resolve()} → check CORPUS_DIR"
CHART_CSV = next(CORPUS_DIR.glob("*.csv"))
print(CHART_CSV)


data\nu-disco-disco\Nu_Disco_Disco.csv


---
## Task 1
**Investigate the use of vocals in your EDMC genre in 2023!**

### 1.1 · In what percentage of the 2023 Top 100 tracks do vocals occur?

In [48]:
# 1.1

# Fetch the top 100 tracks from 2023
df = pd.read_csv(CHART_CSV)
top_tracks = df[(df["rank"] <= 100) & (df["year"] == 2023)]

separator = demucs.api.Separator(model="htdemucs", progress=True, jobs=3)
model = separator.model

STEM_DIR = CORPUS_DIR / "stems"
STEM_DIR.mkdir(parents=True, exist_ok=True)

# Separate stems for all top 100 tracks. Although we only need voice now, other
# stems might come in handy later, and we are generating them anyways.
for id in top_tracks["beatport_track_id"]:
    # Skip missing preview, since we cannot separate stems for this track
    preview = PREVIEW_DIR / f"{id}.mp3"
    if not preview.exists():
        print(f"Missing: {preview}")
        continue

    THIS_STEM = STEM_DIR / str(id)
    THIS_STEM.mkdir(parents=True, exist_ok=True)

    stem_files: dict[str, Path] = {
        "vocals": THIS_STEM / f"{id}_vocals.flac",
        "drums": THIS_STEM / f"{id}_drums.flac",
        "other": THIS_STEM / f"{id}_other.flac",
        "bass": THIS_STEM / f"{id}_bass.flac",
    }

    # Skip if all stems already exists
    if all(file.exists() for file in stem_files.values()):
        print(f"Already found stems for {id}")
        continue

    # If some or all stems do not exist, we delete any existing files
    for file in stem_files.values():
        if file.exists():
            file.unlink()

    try:
        print(f"Separating {id}")
        mix, stems = separator.separate_audio_file(preview)
        for name, stem in stems.items():
            path = stem_files[name]
            tmp = path.with_name(f"{id}_{name}.tmp.flac")
            sf.write(tmp, stem.mean(0).numpy(), separator.samplerate)
            # Ensure only completely written stems are committed
            tmp.rename(path)
    except Exception as ex:
        print(f"Failed: {preview}: {ex}")
        continue

Already found stems for 16924339
Already found stems for 13527555
Already found stems for 16549130
Already found stems for 13601981
Already found stems for 10591853
Already found stems for 16937567
Already found stems for 13061141
Already found stems for 16576721
Already found stems for 16376442
Already found stems for 16157232
Missing: data\nu-disco-disco\previews\15847935.mp3
Already found stems for 13965507
Already found stems for 15363794
Already found stems for 17373224
Already found stems for 17441610
Already found stems for 15469855
Already found stems for 17574049
Already found stems for 17640428
Already found stems for 14980534
Already found stems for 17496391
Already found stems for 16286836
Already found stems for 17444217
Already found stems for 8602163
Already found stems for 17603861
Already found stems for 17714322
Already found stems for 9323977
Already found stems for 17716645
Already found stems for 13702413
Already found stems for 16750749
Already found stems for 137

In [ ]:
def find_vocal_tracks(df: pd.DataFrame) -> pd.DataFrame:
    """
    Determines whether vocals are present in the tracks
    referenced in the data frame based on pre-separated
    vocal stems.

    :param df:  The data frame that contains the tracks

    :returns:   Copy of the input data frame with cols
                "has_preview" and "has_vocals" added.
    """
    vocal_tracks = df.copy()
    vocal_tracks["has_vocals"] = False
    vocal_tracks["has_preview"] = False

    for id in tqdm.tqdm(vocal_tracks["beatport_track_id"]):
        preview = PREVIEW_DIR / f"{id}.mp3"
        stem = STEM_DIR / str(id) / f"{id}_vocals.flac"

        if not preview.exists():
            continue

        # Indicate that preview exists for track
        mask = vocal_tracks["beatport_track_id"] == id
        vocal_tracks.loc[mask, "has_preview"] = True

        if not stem.exists():
            raise Exception(f"Vocal stem for {id} doesn't exist")

        mix, m_sr = librosa.load(preview, sr=44100)
        voc, v_sr = sf.read(stem, dtype="float32")

        if m_sr != v_sr:
            raise Exception(
                f"Mix sample rate is different from vocal stem sample rate: {m_sr} != {v_sr}"
            )

        # Ensure equal length
        n = min(len(mix), len(voc))
        mix, voc = mix[:n], voc[:n]

        rms_mix: npt.NDArray = librosa.feature.rms(y=mix)[0]
        rms_voc: npt.NDArray = librosa.feature.rms(y=voc)[0]

        # Measure vocal level in track in dB
        rel_db = 20 * np.log10((rms_voc + 1e-10) / (rms_mix + 1e-10))

        # Determine if vocals are present in the track.
        # Active level set to at least -15 dB
        # Presence of vocals set to at least 5% active frames
        if (rel_db > -15).sum() / rel_db.size > 0.05:
            vocal_tracks.loc[mask, "has_vocals"] = True

    return vocal_tracks


vocal_tracks_csv = OUTPUT_DIR / "vocal_tracks.csv"
if not vocal_tracks_csv.exists():
    find_vocal_tracks(top_tracks).to_csv(vocal_tracks_csv)

tracks = pd.read_csv(vocal_tracks_csv)

vocals = tracks["has_vocals"] & tracks["has_preview"]
previews = tracks["has_preview"]
print(f"Number of songs with previews: {previews.sum()}")
print(f"Number of songs with vocals: {vocals.sum()}")
print(
    f"Percentage of songs with vocals: {vocals.sum() / previews.sum() * 100:.2f}%"
)

# Random sample of song that should contain vocals
id_voc = tracks[vocals].sample(1)["beatport_track_id"].item()
print("\nThis track should have vocals:")
display(Audio(filename=PREVIEW_DIR / f"{id_voc}.mp3"))

# Random sample of song that should not contain vocals
id_nvoc = tracks[previews & ~vocals].sample(1)["beatport_track_id"].item()
print("This track should not have vocals:")
display(Audio(filename=PREVIEW_DIR / f"{id_nvoc}.mp3"))

Number of songs with previews: 92
Number of songs with vocals: 88
Percentage of songs with vocals: 95.65%

This track should have vocals:


This track should not have vocals:


### 1.2 · What share do the different keys have among the 2023 tracks with vocals?

In [ ]:
# 1.2


### 1.3 · Make it possible to follow by ear why the key could not be detected for certain tracks

In [ ]:
# 1.3


*Embed your audio examples here.*

### 1.4 · Create a results plot and describe the results in one sentence a layperson can understand!

In [ ]:
# 1.4


#### Answer 1.4
*Your answer*

---
## Task 2
**Try to determine the typical use of swing in the drums in your EDMC genre in 2023!**

*By “swing” we mean periodically recurring timing deviations of paired eighth-note onsets following a general pattern (50–66.7 %, triplet swing being the absolute maximum).*

*Tip: such patterns can be observed in the inter-onset tempogram.*

### 2.1 · Create a histogram of the degree of swing determined for the 2023 Top 100 tracks of your corpus

In [ ]:
# 2.1


### 2.2 · Identify the 2023 track with a) the strongest swing and b) the weakest swing

In [ ]:
# 2.2


### 2.3 · Show the logic of your swing detection for these two tracks

In [ ]:
# 2.3


### 2.4 · Describe the results in one sentence a layperson can understand

#### Answer 2.4
*Your answer*

### 2.5 · Check the plausibility of your approach with listening examples

In [ ]:
# 2.5


*Embed your audio examples here.*

---
## Before you upload
- [ ] Name, student ID and genre filled in
- [ ] **Restart Kernel & Run All** runs without errors, all outputs visible
- [ ] Figures labelled (axes, units, title); audio examples embedded where asked